# 구조공학 서버 — 세 번째 단계: 다섯 번째 주차의 검색 보강 생성 체인을 리소스로 노출하기

> [!ref] 강의노트 매핑
> - **Week_07.md §2.7 단계 ② 심화** — 다섯 번째 주차에서 구축한 한국 설계기준 검색 보강 생성 체인을 모델 컨텍스트 프로토콜 리소스로 노출하는 단계입니다.
> - **선행 학습**: 두 번째 단계 노트북을 완료하고, 다섯 번째 주차의 한국 설계기준 검색 보강 생성 노트북에서 코퍼스 구조와 하이브리드 검색기를 이해해 두어야 합니다.

## 학습 목표
이번 단계의 핵심은 **이미 만들어 둔 도메인 자산을 새로운 인터페이스로 재활용**하는 경험을 쌓는 것입니다. 구체적으로 다음 네 가지를 달성합니다.

첫째, 다섯 번째 주차 자산을 그대로 재활용합니다. 다섯 번째 주차에서 만든 한국 설계기준 하이브리드 검색 체인 즉 벡터 검색과 빈도 기반 검색 그리고 상호 순위 융합을 결합한 체인을 그대로 가져와, 모델 컨텍스트 프로토콜 리소스의 백엔드로 사용합니다. 둘째, 임시 구현을 실제 검색으로 교체합니다. 두 번째 단계에서 만든 동적 리소스 템플릿의 정적 사전 구현을 검색 결과로 대체하여, 자유로운 자연어 질의가 가능하도록 만듭니다. 셋째, 새 자유 검색 리소스를 추가합니다. 사용자가 임의의 자연어 질의를 넣어 한국 설계기준 코퍼스 전체를 검색할 수 있는 새 리소스를 등록합니다. 넷째, 도구와 리소스의 차이를 다시 정리합니다. 거대언어모델이 능동적으로 호출하는 도구와 컨텍스트로 주입되는 리소스의 구별을 검색 보강 생성 통합이라는 새로운 관점에서 명확히 합니다.

## 사전 준비 사항
두 번째 단계 노트북을 끝까지 실행하여 누적된 파이썬 모듈 파일이 갱신된 상태여야 합니다. 또한 다섯 번째 주차의 구조공학 검색 보강 생성 노트북에서 다룬 한국 설계기준 코퍼스 구조와 검색기 클래스 구조를 미리 살펴 두면 본 노트북의 코드 재구성 의도를 더 잘 이해할 수 있습니다. 선택적으로 보야지에이아이와 크로마디비 패키지가 설치되어 있다면 실제 벡터 검색을 함께 실행할 수 있지만, 본 노트북은 빈도 기반 단독 대체 구현으로도 완전히 동작하도록 설계되어 있어 외부 의존성 없이 학습이 가능합니다.


## §1. 왜 검색 보강 생성을 도구가 아니라 리소스로 노출하는가?

다섯 번째 주차에서 만든 한국 설계기준 검색 보강 생성 체인은 본질적으로 **함수 호출** 방식이었습니다. 즉 노트북 안의 파이썬 코드에서 검색기 객체의 검색 메서드를 직접 호출하는 형태였죠. 이를 모델 컨텍스트 프로토콜 **리소스** 로 한 번 더 감싸면 다음과 같은 이점을 얻을 수 있습니다.

| 비교 항목 | 함수 호출 방식 (다섯째 주차) | 리소스 방식 (일곱째 주차) |
|---|---|---|
| 호출 주체 | 노트북 안의 파이썬 코드 | 모든 모델 컨텍스트 프로토콜 클라이언트 즉 명령줄 도구나 시각적 검증 도구나 사용자 정의 클라이언트 등 |
| 컨텍스트 주입 시점 | 수동으로 코드에 명시 | 클라이언트가 자동으로 읽기 호출 |
| 재사용 범위 | 단일 노트북 또는 단일 파이썬 프로세스 | 모든 호환 환경에서 네트워크 너머까지 포함 |
| 캐싱 지원 | 별도 구현 필요 | 프로토콜 레벨에서 자동 캐싱 가능 |

> [!tip] 도구와 리소스의 차이를 다시 한 번 명확히
> 도구는 거대언어모델이 "지금 검색이 필요하다"고 스스로 판단하여 능동적으로 호출하는 자산입니다. 따라서 부작용이 있거나 매번 결과가 달라지는 작업에 적합합니다. 리소스는 사용자나 클라이언트 앱이 "이 컨텍스트를 항상 첨부해서 보아라"고 명시적으로 지시할 때 주입되는 자산입니다. 따라서 정적이거나 결정적인 즉 같은 입력에 같은 출력이 보장되는 자산에 적합합니다.

> [!ref] 강의노트 §2.7 단계 ② 심화 인용
> "다섯 번째 주차에서 만든 한국 설계기준 검색 보강 생성 체인을 그대로 가져와 모델 컨텍스트 프로토콜 리소스로 감싼다. 도메인 자산은 한 번 만들어 두면 여러 인터페이스로 반복적으로 노출할 수 있다는 점이 핵심이다. 이는 단순 코드 재사용을 넘어, 도메인 지식을 한 번 정제하면 여러 학습 단계가 서로 연결되는 누적 효과를 만들어 낸다."


## §2. 다섯 번째 주차의 검색 보강 생성 체인 가져오기 — 두 가지 접근 방식

다섯 번째 주차의 노트북은 주피터 노트북 파일이므로 일반적인 파이썬 가져오기 문장으로는 직접 불러올 수 없습니다. 이 문제를 해결하는 두 가지 방법이 있습니다.

첫 번째 대안은 운영 환경에서 권장됩니다. 노트북 변환 도구로 노트북 파일을 파이썬 모듈 파일로 변환한 뒤 가져오기 문장으로 불러옵니다. 실제 배포 환경에서는 노트북을 모듈화하는 것이 일반적인 방식이며, 의존성 관리와 단위 시험이 모두 매끄러워집니다.

두 번째 대안은 학습용으로 권장됩니다. 핵심 클래스 즉 한국 설계기준 코퍼스 자료와 검색기 클래스를 본 노트북에서 다시 정의합니다. 코드 중복이 발생한다는 단점이 있지만, 학습 흐름이 자연스럽고 셀 단위로 동작을 추적하기 쉽다는 큰 장점이 있습니다.

본 노트북에서는 학습 목적이므로 두 번째 대안을 채택하여 다섯 번째 주차의 코드를 **핵심만 추려 재구성** 합니다. 원본 코드 전체는 다섯 번째 주차의 구조공학 검색 보강 생성 노트북에서 확인할 수 있습니다.

> [!finding] 학습 의도 — 코드 중복으로 얻는 통찰
> 같은 코드를 두 번 적는 것은 일반적으로 권장되지 않지만, 여기서는 학생이 "도메인 서버는 결국 다섯 번째 주차의 검색 보강 생성 위에 래퍼 데코레이터만 추가하면 된다"는 사실을 시각적으로 확인하는 것이 더 중요합니다. 두 노트북을 나란히 펴 놓고 비교해 보세요. 그러면 모델 컨텍스트 프로토콜의 본질이 단순한 인터페이스 추상화 위에 도메인 자산을 노출하는 일임을 한눈에 알 수 있습니다.


In [ ]:
# Setup — KDS 코퍼스 (W05 S4_07_structural_rag.ipynb의 KDS_SECTIONS 인용)
import json
import math
import re
from collections import Counter
from mcp.server.fastmcp import FastMCP
from pydantic import Field

# W05 §1 KDS_SECTIONS — 8개 조항 샘플
KDS_SECTIONS = [
    {"id": "KDS-41-10-15-010", "title": "콘크리트 설계기준강도",
     "text": "KDS 41 10 15 (010): 콘크리트의 설계기준강도 fck는 최소 21 MPa 이상이어야 한다."},
    {"id": "KDS-41-10-15-020", "title": "철근 재료 기준",
     "text": "KDS 41 10 15 (020): 이형철근의 항복강도 fy는 SD400 또는 SD500을 표준으로 한다."},
    {"id": "KDS-41-10-20-010", "title": "보 설계 — 휨",
     "text": "KDS 41 10 20 (010): RC 보의 최소 인장철근비는 max(0.25*sqrt(fck)/fy, 1.4/fy) 이상이어야 한다."},
    {"id": "KDS-41-10-20-020", "title": "보 설계 — 전단",
     "text": "KDS 41 10 20 (020): 전단보강 철근(스터럽)의 최대 간격은 d/2 이하로 한다."},
    {"id": "KDS-41-10-20-030", "title": "기둥 설계",
     "text": "KDS 41 10 20 (030): 기둥의 최소 단면치수는 300mm 이상이어야 한다."},
    {"id": "KDS-41-12-00-010", "title": "하중 조합",
     "text": "KDS 41 12 00 (010): 하중조합은 1.2D + 1.6L (기본), 1.2D + 1.0L + 1.0E (지진)."},
    {"id": "KDS-41-10-10-020", "title": "고정하중",
     "text": "KDS 41 10 10 (020): 철근콘크리트 25 kN/m³, 보통중량 콘크리트 24 kN/m³."},
    {"id": "KDS-41-10-10-030", "title": "적재하중",
     "text": "KDS 41 10 10 (030): 주거용 2.0 kN/m², 사무실 2.5 kN/m², 상점 4.0 kN/m²."},
]

print(f"✅ KDS 코퍼스 {len(KDS_SECTIONS)}개 조항 로드")


## §3. 경량 빈도 기반 검색기 — 검색 보강 생성 대체 구현

실제 검색 보강 생성 환경 즉 보야지에이아이 임베딩 라이브러리와 크로마디비 벡터 저장소를 갖추지 못한 학습용 환경에서도 본 노트북이 동작할 수 있도록, **외부 라이브러리에 의존하지 않는 빈도 기반 단독 검색기** 를 대체 구현으로 만들어 둡니다. 이는 다섯 번째 주차에서 만든 빈도 기반 색인 클래스의 핵심 알고리즘만 추려서 재구성한 것입니다.

빈도 기반 검색기는 정보 검색 분야의 고전적인 가중치 함수로, 다음 세 가지 요소를 결합하여 점수를 산정합니다. 첫째 요소는 단어 빈도로, 어떤 단어가 문서 안에 얼마나 자주 등장하는가를 측정합니다. 자주 등장할수록 점수가 올라가지만 무한정 올라가지는 않도록 포화 함수가 적용됩니다. 둘째 요소는 역문서 빈도로, 어떤 단어가 전체 코퍼스에서 얼마나 희귀한가를 측정합니다. 흔한 단어일수록 가중치가 낮아지고, 희귀한 단어일수록 차별성이 높아 점수에 큰 영향을 줍니다. 셋째 요소는 문서 길이 정규화로, 긴 문서가 단순히 길다는 이유로 점수가 부풀려지지 않도록 평균 문서 길이로 정규화합니다.

벡터 검색이 의미론적 유사성을 잡아낸다면, 빈도 기반 검색은 정확한 단어 매칭에 강점이 있어 **두 방식을 결합한 하이브리드 검색** 이 일반적으로 가장 좋은 성능을 보이는 것으로 알려져 있습니다.


In [ ]:
# W05 §2 BM25Index 핵심 재구성 (voyageai 미설치 환경 대비 fallback)
class SimpleBM25:
    """W05 BM25Index의 경량 버전 — 외부 라이브러리 0개."""

    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs, self.ids = [], []
        self.doc_lens, self.avg_len = [], 0.0
        self.df = Counter()
        self.tfs = []

    def _tok(self, text):
        return re.split(r"\W+", text.lower())

    def add(self, sections):
        for s in sections:
            tokens = [t for t in self._tok(s["text"]) if t]
            self.docs.append(s["text"])
            self.ids.append(s["id"])
            self.doc_lens.append(len(tokens))
            tf = Counter(tokens)
            self.tfs.append(tf)
            for t in set(tokens):
                self.df[t] += 1
        self.avg_len = sum(self.doc_lens) / max(len(self.doc_lens), 1)

    def search(self, query, k=3):
        n = len(self.docs)
        if n == 0:
            return []
        tokens = [t for t in self._tok(query) if t]
        scores = []
        for i in range(n):
            score = 0.0
            for t in tokens:
                tf = self.tfs[i].get(t, 0)
                df = self.df.get(t, 0)
                idf = math.log((n - df + 0.5) / (df + 0.5) + 1)
                norm = self.k1 * (1 - self.b + self.b * self.doc_lens[i] / self.avg_len)
                score += idf * tf * (self.k1 + 1) / (tf + norm + 1e-9)
            scores.append(score)
        ranked = sorted(zip(self.ids, self.docs, scores), key=lambda x: -x[2])
        return [{"id": i, "text": t, "score": s} for i, t, s in ranked[:k] if s > 0]


bm25 = SimpleBM25()
bm25.add(KDS_SECTIONS)

# 동작 확인
hits = bm25.search("최소 철근비 보", k=3)
for h in hits:
    print(f"  [{h['id']}] score={h['score']:.3f}  {h['text'][:50]}...")


## §4. 검색 진입점 함수 정의 — 다섯 번째 주차 호환 인터페이스

다섯 번째 주차의 검색기 메서드 시그니처를 그대로 모방한 함수를 만듭니다. 이렇게 함수 이름과 인자 형식을 맞춰 두면, 나중에 운영 환경으로 옮길 때 함수 본체만 교체하고 호출부는 그대로 둘 수 있다는 장점이 있습니다.

본 학습용 노트북에서는 빈도 기반 단독으로 동작하지만, 실제 운영 환경에서는 다음과 같이 교체합니다. 먼저 보야지에이아이 임베딩 라이브러리로 한국 설계기준 코퍼스 전체를 벡터로 변환하여 크로마디비에 저장합니다. 그다음 질의가 들어오면 벡터 검색과 빈도 기반 검색을 동시에 수행한 뒤, 상호 순위 융합 알고리즘으로 두 결과를 결합합니다. 마지막으로 최종 상위 케이 개 문서를 반환합니다.

> [!ref] 다섯 번째 주차의 검색 보강 생성 노트북 인용
> 본격적인 하이브리드 검색기는 벡터 색인과 빈도 기반 색인 두 가지를 모두 받는 검색기 객체를 생성한 뒤 검색 메서드를 호출하는 형태로 작동합니다. 본 노트북은 이 인터페이스를 학습용으로 단순화한 버전이며, 함수 본체만 교체하면 곧바로 운영 환경으로 옮길 수 있도록 설계되었습니다.


In [ ]:
def hybrid_retrieve(query: str, k: int = 3) -> list[dict]:
    """W05 RAG 체인 호환 인터페이스 — 현재는 BM25 fallback.

    프로덕션에서는 다음으로 교체:
        from S4_07_structural_rag import Retriever  # nbconvert 후
        retriever = Retriever(vector_index, bm25_index)
        return retriever.search(query, top_k=k)
    """
    return bm25.search(query, k=k)


# 테스트: 다양한 쿼리
for q in ["최소 철근비", "사무실 적재하중", "지진 하중 조합"]:
    print(f"\n쿼리: '{q}'")
    for h in hybrid_retrieve(q, k=2):
        print(f"  [{h['id']}] {h['text'][:60]}...")


## §5. 서버 인스턴스 생성과 새 검색 보강 생성 리소스 등록

이제 빈도 기반 검색기를 백엔드로 삼는 두 가지 검색 보강 생성 리소스를 등록합니다.

첫 번째는 자유 검색 리소스입니다. 사용자가 임의의 자연어 질의를 넣으면, 그 질의에 가장 관련성 높은 한국 설계기준 조항 상위 세 개를 제이슨 형식으로 반환합니다. 예를 들어 자유 검색 리소스에 "최소 철근비"라는 질의를 넣어 호출하면, 코퍼스에서 철근비 관련 조항이 점수 순으로 정렬되어 반환됩니다.

두 번째는 조문 식별자 기반 검색 리소스입니다. 두 번째 단계 노트북에서 만든 임시 구현을 검색 보강 생성 기반으로 교체하는 의미가 있습니다. 조문 식별자 예를 들어 사 점 삼이나 사 점 사 같은 값을 검색 질의로 변환하여 가장 관련 높은 조항 한 개를 반환합니다. 이는 임시 구현이 사전 룩업으로 미리 등록된 식별자만 처리할 수 있었던 한계를 넘어, 새로운 식별자가 들어와도 코퍼스에서 가장 가까운 조항을 자동으로 찾아 반환할 수 있게 해 줍니다.


In [ ]:
mcp = FastMCP("StructuralMCP", log_level="ERROR")


# 새 리소스 1 — 자유 검색 (templated)
@mcp.resource("kds://search/{query}", mime_type="application/json")
def search_kds(query: str) -> str:
    """KDS 코퍼스에서 query 관련 조항을 RAG로 검색합니다."""
    results = hybrid_retrieve(query, k=3)
    return json.dumps({
        "query": query,
        "k": 3,
        "results": results,
    }, ensure_ascii=False, indent=2)


# 새 리소스 2 — 기존 section_id 리소스를 RAG 기반으로 교체
@mcp.resource("kds://41-17-00/{section_id}", mime_type="text/plain")
def get_kds_section_rag(section_id: str) -> str:
    """KDS 조문 ID로 검색 — Stage 2 placeholder를 RAG로 대체."""
    # section_id (예: "4.3") → KDS 코퍼스에서 가장 관련 높은 조항
    results = hybrid_retrieve(f"KDS 41 10 20 {section_id}", k=1)
    if not results:
        return f"Section {section_id}: no match in KDS corpus"
    top = results[0]
    return f"[{top['id']}] {top['text']}"


print("✅ RAG 기반 리소스 2개 등록")
print("  - kds://search/{query}     (자유 검색)")
print("  - kds://41-17-00/{section_id}  (조문 ID → RAG)")


## §6. 검증 단계 — RAG 리소스 호출로 동작 확인

지금까지 등록한 RAG 리소스 두 개가 의도대로 동작하는지 실제 호출로 확인합니다. 자유 검색의 경우 한국어 자연어 질의("휨 철근비", "사무실 적재하중")를 넣어 BM25가 적절한 KDS 조항을 찾아내는지 봅니다. 조문 ID 검색의 경우 "4.3" 같은 식별자를 넣어 휨 부재 관련 조항이 반환되는지 확인합니다.

여기서 BM25는 토큰 수준 매칭에 의존하므로, 질의에 KDS 조항 본문에 등장하는 단어가 포함되어 있을 때 가장 좋은 결과를 보입니다. 운영 환경의 벡터 + BM25 하이브리드 검색은 이런 한계를 의미론적 유사성으로 보완합니다.


In [ ]:
import asyncio

async def test_rag_resources():
    # 자유 검색
    r1 = await mcp.read_resource("kds://search/휨 철근비")
    print("=== kds://search/휨 철근비 ===")
    print(str(r1)[:400])

    # 조문 ID 검색
    r2 = await mcp.read_resource("kds://41-17-00/4.3")
    print("\n=== kds://41-17-00/4.3 ===")
    print(r2)

    r3 = await mcp.read_resource("kds://search/사무실 적재하중")
    print("\n=== kds://search/사무실 적재하중 ===")
    print(str(r3)[:400])

await test_rag_resources()


## §7. 파이썬 모듈 파일 갱신 저장 — 검색 보강 생성 통합 버전

지금까지 누적된 모든 자산을 하나의 파이썬 모듈 파일로 다시 씁니다. 누적 결과를 정리하면 다음과 같습니다.

지금 시점까지 본 트랙에서 만들어진 자산은 다음과 같이 분류됩니다. 도구는 한 개로 첫 번째 단계에서 정의한 무근 콘크리트 압축강도 검토 함수입니다. 정적 리소스는 세 개로 두 번째 단계에서 정의한 한국 설계기준 요약과 콘크리트 표와 철근 표입니다. 검색 보강 생성 리소스는 두 개로 본 노트북에서 새로 정의한 자유 검색 리소스와 조문 식별자 기반 검색 리소스입니다.

이 파일이 다음 단계 노트북의 입력이 되며, 거기서는 마이다스 시빌의 입력 파일을 읽어 절점과 부재를 추출하는 도구가 추가됩니다. 이렇게 단계마다 누적된 산출물이 다음 단계의 출발점이 되는 패턴이 일관되게 유지된다는 점에 주목하세요.


In [ ]:
STAGE3_SOURCE = '''"""structural-mcp — Stage 3 (도구 1 + 정적 리소스 3 + RAG 리소스 2)

자동 생성: S6_st03_kds_rag_resource.ipynb
W05 KDS RAG 체인 통합 — kds://search/{query}, kds://41-17-00/{section_id}
다음 단계 S6_st04에서 Midas .mgt 파서 도구가 추가됩니다.
"""
import json
import math
import re
from collections import Counter
from mcp.server.fastmcp import FastMCP
from pydantic import Field

# ── KDS 코퍼스 (W05 S4_07_structural_rag.ipynb 인용) ──
KDS_SECTIONS = [
    {"id": "KDS-41-10-15-010", "text": "콘크리트 설계기준강도 fck는 최소 21 MPa 이상."},
    {"id": "KDS-41-10-15-020", "text": "이형철근 항복강도 fy는 SD400 또는 SD500."},
    {"id": "KDS-41-10-20-010", "text": "RC 보 최소 인장철근비 max(0.25*sqrt(fck)/fy, 1.4/fy)."},
    {"id": "KDS-41-10-20-020", "text": "전단보강 철근 간격 d/2 이하."},
    {"id": "KDS-41-10-20-030", "text": "기둥 최소 단면치수 300mm 이상."},
    {"id": "KDS-41-12-00-010", "text": "하중조합 1.2D+1.6L (기본), 1.2D+1.0L+1.0E (지진)."},
    {"id": "KDS-41-10-10-020", "text": "고정하중 철근콘크리트 25 kN/m³."},
    {"id": "KDS-41-10-10-030", "text": "적재하중 주거용 2.0 kN/m², 사무실 2.5 kN/m²."},
]


# ── BM25 검색기 (W05 §2 인용) ──
class SimpleBM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs, self.ids = [], []
        self.doc_lens, self.avg_len = [], 0.0
        self.df = Counter()
        self.tfs = []

    def _tok(self, text):
        return [t for t in re.split(r"\\W+", text.lower()) if t]

    def add(self, sections):
        for s in sections:
            tokens = self._tok(s["text"])
            self.docs.append(s["text"]); self.ids.append(s["id"])
            self.doc_lens.append(len(tokens))
            self.tfs.append(Counter(tokens))
            for t in set(tokens): self.df[t] += 1
        self.avg_len = sum(self.doc_lens) / max(len(self.doc_lens), 1)

    def search(self, query, k=3):
        n = len(self.docs)
        if n == 0: return []
        toks = self._tok(query)
        scored = []
        for i in range(n):
            sc = 0.0
            for t in toks:
                tf = self.tfs[i].get(t, 0); df = self.df.get(t, 0)
                idf = math.log((n - df + 0.5) / (df + 0.5) + 1)
                norm = self.k1 * (1 - self.b + self.b * self.doc_lens[i] / self.avg_len)
                sc += idf * tf * (self.k1 + 1) / (tf + norm + 1e-9)
            scored.append((self.ids[i], self.docs[i], sc))
        scored.sort(key=lambda x: -x[2])
        return [{"id": i, "text": t, "score": s} for i, t, s in scored[:k] if s > 0]


_bm25 = SimpleBM25(); _bm25.add(KDS_SECTIONS)


def hybrid_retrieve(query: str, k: int = 3):
    return _bm25.search(query, k=k)


# ── MCP Server ──
mcp = FastMCP("StructuralMCP", log_level="ERROR")


@mcp.tool()
def check_concrete_strength(
    fck: float = Field(description="콘크리트 fck (MPa)"),
    Pu: float = Field(description="소요 축력 (kN)"),
    Ag: float = Field(description="기둥 단면적 (mm²)"),
) -> str:
    """무근 콘크리트 압축강도 검토 (KDS 41 17 00)."""
    phi = 0.65; Pn = 0.85 * fck * Ag / 1000.0; phi_Pn = phi * Pn
    DCR = Pu / phi_Pn if phi_Pn > 0 else float("inf")
    return json.dumps({"phi_Pn_kN": round(phi_Pn, 2), "Pu_kN": Pu,
                       "DCR": round(DCR, 3),
                       "check": "OK" if DCR <= 1.0 else "NG"},
                      indent=2, ensure_ascii=False)


@mcp.resource("kds://41-17-00/summary", mime_type="application/json")
def get_kds_summary():
    return json.dumps({"flexure": {"phi": 0.85}, "shear": {"phi": 0.75},
                       "axial": {"phi": 0.65}}, indent=2, ensure_ascii=False)


@mcp.resource("data://materials/concrete-table", mime_type="application/json")
def get_concrete_table():
    return json.dumps({"C24": {"fck": 24}, "C27": {"fck": 27},
                       "C30": {"fck": 30}, "C35": {"fck": 35},
                       "C40": {"fck": 40}}, indent=2, ensure_ascii=False)


@mcp.resource("data://materials/rebar-table", mime_type="application/json")
def get_rebar_table():
    return json.dumps({"D10": 71.3, "D13": 126.7, "D16": 198.6,
                       "D19": 286.5, "D22": 387.1, "D25": 506.7,
                       "D29": 642.4, "D32": 794.2}, indent=2, ensure_ascii=False)


@mcp.resource("kds://search/{query}", mime_type="application/json")
def search_kds(query: str):
    """W05 RAG 체인을 MCP 리소스로 노출."""
    return json.dumps({"query": query, "results": hybrid_retrieve(query, k=3)},
                      ensure_ascii=False, indent=2)


@mcp.resource("kds://41-17-00/{section_id}", mime_type="text/plain")
def get_kds_section_rag(section_id: str):
    results = hybrid_retrieve(f"KDS 41 10 20 {section_id}", k=1)
    if not results:
        return f"Section {section_id}: no match"
    return f"[{results[0]['id']}] {results[0]['text']}"


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("structural_mcp.py", "w", encoding="utf-8") as f:
    f.write(STAGE3_SOURCE)

print(f"✅ structural_mcp.py Stage 3 저장 ({len(STAGE3_SOURCE)} bytes)")
print(f"   - 도구 1, 정적 리소스 3, RAG 리소스 2")


## §8. 다음 단계 안내 — 네 번째 단계 노트북으로 이어집니다

> [!action] 다음에 학습할 노트북
> 네 번째 단계 노트북에서는 마이다스 시빌의 텍스트 입력 형식을 파싱하여 절점과 부재와 하중 정보를 도구로 노출합니다. 이는 구조해석 입력 데이터를 거대언어모델이 직접 읽고 검토하는 첫 사례이며, 본 트랙에서 가장 도전적인 부분입니다. 텍스트 기반 입력 파일이라는 특성을 잘 활용하면, 일반 대화 환경에서 자연어로 "이 모델 파일을 읽어 절점 수와 부재 수를 알려 줘" 같은 요청을 처리할 수 있게 됩니다.

> [!finding] 세 단계까지 누적된 자산 정리
> 지금까지 만들어진 도메인 자산을 정리하면 다음과 같습니다. 도구는 한 개로 무근 콘크리트 압축강도 검토 함수입니다. 정적 리소스는 세 개로 한국 콘크리트구조 설계기준 요약, 콘크리트 강도 등급별 물성치 표, 이형철근 규격별 단면적 표입니다. 검색 보강 생성 기반 동적 리소스도 두 개가 있는데, 자유 검색용 리소스와 조문 식별자 기반 검색 리소스입니다.
>
> 본 단계의 핵심 학습 성과는 다섯 번째 주차에서 만든 검색 보강 생성 체인을 **리소스 래퍼만 추가하여 그대로 재활용** 했다는 점입니다. 도메인 자산은 한 번 만들어 두면 새로운 인터페이스로 반복적으로 노출할 수 있다는 누적 효과를 직접 체험했습니다. 이는 단지 코드 재사용 차원이 아니라, 도메인 지식을 한 번 정제하면 여러 주차의 학습 결과가 서로 연결되어 시너지를 만들어 낸다는 보다 깊은 통찰을 담고 있습니다.
